# TSDF Mesh Visualisation

Interactive 3-D viewer for the three meshes produced by `tsdf_meshify.py`.

TSDF meshes can be very large (millions of triangles), so each mesh is decimated  
to `MAX_DISPLAY_FACES` for the interactive viewer.  The full-resolution meshes  
are used for Chamfer evaluation.

**Set the paths in the cell below, then run all cells.**

In [ ]:
# ── CONFIGURE ───────────────────────────────────────────────────────────────
# Root output directory for one object (the same path you passed to tsdf_meshify.py)
# OBJECT_ROOT = "output/table/Table_34610"
# OBJECT_ROOT = "/home/stone/dev/projects/articulate/pm_dataset_generator/gt_meshes/Bucket_100443"  # for GT
OBJECT_ROOT = "/home/stone/dev/projects/articulate/pm_dataset_generator/partnet-mobility-video-dataset/Table_34610"  # for GT

# Maximum number of triangles shown in the interactive viewer.
# Reduce further (e.g. 20_000) if the viewer feels slow.
MAX_DISPLAY_FACES = 50_000

# Optional: path to a ground-truth mesh for Chamfer evaluation (set None to skip)
GT_MESH = None  # e.g. "/path/to/Table_34610/gt_mesh.ply"
# GT_MESH = "/home/stone/dev/projects/articulate/pm_dataset_generator/gt_meshes/Table_34610/gt_mesh.ply"  # for GT visualization

# Number of surface samples for Chamfer distance
N_CHAMFER_SAMPLES = 10_000
# ────────────────────────────────────────────────────────────────────────────

In [7]:
import os
import sys
from pathlib import Path

import numpy as np
import trimesh
import plotly.graph_objects as go
from plotly.subplots import make_subplots

MESH_DIR = os.path.join(OBJECT_ROOT, "meshes")
# MESH_PATHS = {
#     "static":  os.path.join(MESH_DIR, "tsdf_mesh_static.ply"),
#     "dynamic": os.path.join(MESH_DIR, "tsdf_mesh_dynamic.ply"),
#     "full":    os.path.join(MESH_DIR, "tsdf_mesh_full.ply"),
# }
MESH_PATHS = {
    "static":  os.path.join(OBJECT_ROOT, "multiview_static", "gt_static.ply"),
    "dynamic": os.path.join(OBJECT_ROOT, "multiview_static", "gt_dynamic.ply"),
    "full":    os.path.join(OBJECT_ROOT, "multiview_static", "gt_full.ply"),
}

print(f"{'Mesh':10s}  {'Size':>8s}  Path")
print("-" * 72)
for label, p in MESH_PATHS.items():
    if os.path.isfile(p):
        sz = f"{os.path.getsize(p)/1e6:.1f} MB"
    else:
        sz = "MISSING"
    print(f"{label:10s}  {sz:>8s}  {p}")

Mesh            Size  Path
------------------------------------------------------------------------
static       MISSING  output/table/Table_34610/multiview_static/gt_static.ply
dynamic      MISSING  output/table/Table_34610/multiview_static/gt_dynamic.ply
full         MISSING  output/table/Table_34610/multiview_static/gt_full.ply


In [3]:
# ── Load full-resolution meshes ──────────────────────────────────────────────

def load_mesh(path: str) -> trimesh.Trimesh:
    m = trimesh.load(path, force="mesh", process=False)
    if isinstance(m, trimesh.Scene):
        m = trimesh.util.concatenate(
            [g for g in m.dump() if isinstance(g, trimesh.Trimesh)]
        )
    return m

meshes = {}
for label, p in MESH_PATHS.items():
    if os.path.isfile(p):
        meshes[label] = load_mesh(p)

print(f"{'Mesh':10s}  {'Vertices':>12s}  {'Faces':>12s}  {'Watertight':>10s}")
print("-" * 56)
for label, m in meshes.items():
    print(f"{label:10s}  {len(m.vertices):>12,d}  {len(m.faces):>12,d}  {str(m.is_watertight):>10s}")

Mesh            Vertices         Faces  Watertight
--------------------------------------------------------


In [4]:
# ── Decimate meshes for the interactive viewer ───────────────────────────────
#
# TSDF meshes easily exceed 1 M triangles.  We decimate to MAX_DISPLAY_FACES
# using quadric error metrics (fast-simplification); the full-res meshes are
# kept in `meshes` for Chamfer evaluation.

display_meshes = {}
for label, m in meshes.items():
    if len(m.faces) > MAX_DISPLAY_FACES:
        m_disp = m.simplify_quadric_decimation(face_count=MAX_DISPLAY_FACES)
        print(f"{label:10s}  {len(m.faces):>12,d} → {len(m_disp.faces):>8,d} faces (decimated)")
    else:
        m_disp = m
        print(f"{label:10s}  {len(m.faces):>12,d}  (no decimation needed)")
    display_meshes[label] = m_disp

In [5]:
# ── Colour palette ───────────────────────────────────────────────────────────

PALETTE = {
    "static":  dict(color="#4a90d9", name="Static  (label 0)"),
    "dynamic": dict(color="#e05c45", name="Dynamic (label ≥1)"),
    "full":    dict(color="#4caf7d", name="Full"),
}

_SCENE_CFG = dict(
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    zaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    bgcolor="rgb(15,15,20)",
    aspectmode="data"
)

In [6]:
# ── Interactive 3-panel viewer ───────────────────────────────────────────────

active_labels = [l for l in ["static", "dynamic", "full"] if l in display_meshes]
n_cols = len(active_labels)

fig = make_subplots(
    rows=1, cols=n_cols,
    specs=[[{"type": "scene"}] * n_cols],
    subplot_titles=[PALETTE[l]["name"] for l in active_labels],
    horizontal_spacing=0.02,
)

for col, label in enumerate(active_labels, start=1):
    m    = display_meshes[label]
    v, f = m.vertices, m.faces
    cfg  = PALETTE[label]
    fig.add_trace(
        go.Mesh3d(
            x=v[:, 0], y=v[:, 1], z=v[:, 2],
            i=f[:, 0], j=f[:, 1], k=f[:, 2],
            color=cfg["color"],
            opacity=1.0,
            name=cfg["name"],
            lighting=dict(ambient=0.4, diffuse=0.6, specular=0.2, roughness=0.5),
            lightposition=dict(x=2, y=5, z=5),
            flatshading=False,
            showscale=False,
        ),
        row=1, col=col,
    )
    fig.update_scenes(
        {**_SCENE_CFG, "camera": dict(eye=dict(x=1.5, y=1.5, z=1.2))},
        row=1, col=col,
    )

_name = Path(OBJECT_ROOT).name
fig.update_layout(
    title=dict(text=f"<b>{_name}</b> — TSDF meshes ({MAX_DISPLAY_FACES:,} faces/mesh)",
               font=dict(size=16, color="white")),
    paper_bgcolor="rgb(15,15,20)",
    font=dict(color="white"),
    height=560,
    margin=dict(l=0, r=0, t=60, b=0),
    showlegend=False,
)
fig.show()

ValueError: 
The 'cols' argument to make_subplots must be an int greater than 0.
    Received value of type <class 'int'>: 0

In [ ]:
# ── Overlay viewer: all three meshes in one scene ───────────────────────────
#
# Useful for checking spatial alignment between static and dynamic parts.

fig2 = go.Figure()
for label, m in display_meshes.items():
    v, f = m.vertices, m.faces
    cfg  = PALETTE[label]
    fig2.add_trace(go.Mesh3d(
        x=v[:, 0], y=v[:, 1], z=v[:, 2],
        i=f[:, 0], j=f[:, 1], k=f[:, 2],
        color=cfg["color"],
        opacity=0.45,
        name=cfg["name"],
        lighting=dict(ambient=0.4, diffuse=0.6),
        flatshading=False,
    ))

_name = Path(OBJECT_ROOT).name
fig2.update_layout(
    title=dict(text=f"<b>{_name}</b> — overlay", font=dict(size=15, color="white")),
    paper_bgcolor="rgb(15,15,20)",
    scene={**_SCENE_CFG, "camera": dict(eye=dict(x=1.5, y=1.5, z=1.2))},
    font=dict(color="white"),
    height=560,
    margin=dict(l=0, r=0, t=60, b=0),
    legend=dict(x=0.01, y=0.98, bgcolor="rgba(0,0,0,0)", font=dict(color="white")),
)
fig2.show()

---
## Chamfer Evaluation

### Self-consistency check
Chamfer distances between the three meshes (static ↔ full, dynamic ↔ full).  
No GT mesh required — useful for verifying the pipeline end-to-end.

Full-resolution meshes are sampled (point clouds), so the metric is unaffected by decimation.

Set `GT_MESH` at the top of the notebook to also compare against a reference mesh.

In [ ]:
# ── Chamfer helpers ──────────────────────────────────────────────────────────

from scipy.spatial import cKDTree

def sample_surface(mesh: trimesh.Trimesh, n: int) -> np.ndarray:
    pts, _ = trimesh.sample.sample_surface(mesh, n)
    return pts

def chamfer_distance(pts_a: np.ndarray, pts_b: np.ndarray, thresh: float = 0.05) -> dict:
    """Symmetric Chamfer distance + F-score at given distance threshold."""
    tree_a = cKDTree(pts_a)
    tree_b = cKDTree(pts_b)
    d_ab, _ = tree_b.query(pts_a)   # nearest-neighbour distances A→B
    d_ba, _ = tree_a.query(pts_b)   # nearest-neighbour distances B→A
    cd        = float(np.mean(d_ab ** 2) + np.mean(d_ba ** 2))
    precision = float(np.mean(d_ab < thresh))
    recall    = float(np.mean(d_ba < thresh))
    f1 = (2 * precision * recall / (precision + recall)
          if (precision + recall) > 0 else 0.0)
    return {"chamfer_distance": cd, "precision": precision,
            "recall": recall, "f1": f1}

In [ ]:
# ── Self-consistency Chamfer distances ──────────────────────────────────────

pairs = [
    ("static  → full",    "static",  "full"),
    ("dynamic → full",    "dynamic", "full"),
    ("static  → dynamic", "static",  "dynamic"),
]

cd_results = {}
for title, a, b in pairs:
    if not (a in meshes and b in meshes):
        print(f"{title}: one or both meshes not loaded — skipped")
        continue
    pts_a = sample_surface(meshes[a], N_CHAMFER_SAMPLES)
    pts_b = sample_surface(meshes[b], N_CHAMFER_SAMPLES)
    r = chamfer_distance(pts_a, pts_b)
    cd_results[title] = r
    print(f"{title:25s}  CD={r['chamfer_distance']:.6f}  "
          f"prec={r['precision']:.4f}  recall={r['recall']:.4f}  F1={r['f1']:.4f}")

print("")
print("Note: self-consistency CD measures part-mesh overlap, not reconstruction accuracy.")

static  → full             CD=0.006686  prec=0.9447  recall=0.7290  F1=0.8230
dynamic → full             CD=0.011867  prec=0.9352  recall=0.5431  F1=0.6872
static  → dynamic          CD=0.029990  prec=0.2959  recall=0.3581  F1=0.3240

Note: self-consistency CD measures part-mesh overlap, not reconstruction accuracy.


In [ ]:
# ── Chamfer vs GT mesh (skip if GT_MESH is None) ─────────────────────────────

if GT_MESH is None:
    print("GT_MESH is not set — skipping GT evaluation.")
    print("Set GT_MESH at the top of the notebook to enable this cell.")
else:
    gt = load_mesh(GT_MESH)
    pts_gt = sample_surface(gt, N_CHAMFER_SAMPLES)
    print(f"GT mesh: {GT_MESH}")
    print(f"{'Mesh':10s}  {'CD':>12s}  {'Precision':>10s}  {'Recall':>8s}  {'F1':>8s}")
    print("-" * 58)
    for label in ["full", "static", "dynamic"]:
        if label not in meshes:
            print(f"{label:10s}  (missing)")
            continue
        pts_pred = sample_surface(meshes[label], N_CHAMFER_SAMPLES)
        r = chamfer_distance(pts_pred, pts_gt)
        print(f"{label:10s}  {r['chamfer_distance']:>12.6f}  "
              f"{r['precision']:>10.4f}  {r['recall']:>8.4f}  {r['f1']:>8.4f}")

GT_MESH is not set — skipping GT evaluation.
Set GT_MESH at the top of the notebook to enable this cell.


In [11]:
# ── Error map: full mesh sample coloured by distance² to GT (needs GT_MESH) ──

if GT_MESH is None:
    print("GT_MESH is not set — skipping error-map visualisation.")
else:
    gt = load_mesh(GT_MESH)
    pts_gt   = sample_surface(gt,              N_CHAMFER_SAMPLES)
    pts_full = sample_surface(meshes["full"],  N_CHAMFER_SAMPLES)

    tree_gt  = cKDTree(pts_gt)
    dists, _ = tree_gt.query(pts_full)
    err      = dists ** 2

    fig_err = go.Figure(go.Scatter3d(
        x=pts_full[:, 0], y=pts_full[:, 1], z=pts_full[:, 2],
        mode="markers",
        marker=dict(
            size=2,
            color=err,
            colorscale="Inferno",
            colorbar=dict(title="dist²"),
            showscale=True,
        ),
    ))
    _name = Path(OBJECT_ROOT).name
    fig_err.update_layout(
        title=f"<b>{_name}</b> — full mesh vs GT, per-point error map",
        paper_bgcolor="rgb(15,15,20)",
        scene=_SCENE_CFG,
        font=dict(color="white"),
        height=560,
        margin=dict(l=0, r=0, t=60, b=0),
    )
    fig_err.show()

GT_MESH is not set — skipping error-map visualisation.
